# Code Initialization

In [1]:
from qiskit import QuantumCircuit, transpile, QuantumRegister
from qiskit_aer import AerSimulator, QasmSimulator, Aer
# from qiskit_ibm_runtime import QiskitRuntimeService, Session, Sampler, Estimator, Options
from qiskit_ibm_runtime import QiskitRuntimeService, Session, Options
from qiskit_ibm_runtime import Sampler, SamplerV2
from qiskit.visualization import plot_histogram
from qiskit.circuit.classical import expr

from datetime import datetime
import mysql.connector
import networkx as nx
import matplotlib.pyplot as plt
# plt.rcParams.update({'font.size': 14})

import numpy as np
import pandas as pd
import seaborn as sns
import re
from qiskit_aer.noise import (NoiseModel, QuantumError, ReadoutError, reset_error,
    pauli_error, depolarizing_error, thermal_relaxation_error)
import json

from qiskit_ibm_runtime.fake_provider import fake_backend

import copy
from qiskit.visualization import plot_histogram, plot_state_city
import qiskit.quantum_info as qi
from qiskit.qasm2 import dumps
from qiskit.visualization import plot_circuit_layout

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import RZZGate, RZGate, XGate, IGate
from qiskit.converters import circuit_to_dag, dag_to_circuit

from qiskit.transpiler.passes import ALAPScheduleAnalysis, ASAPScheduleAnalysis, PadDynamicalDecoupling, PadDelay
from qiskit.transpiler import PassManager
import numpy as np
from qiskit.qasm2 import dumps
from random import randint

import json
from qiskit_ibm_runtime import RuntimeEncoder, RuntimeDecoder

import mthree
import stim
import time


CB_color_cycle = [
    '#006BA4',  # Blue
    '#FF800E',  # Orange
    '#ABABAB',  # Gray
    '#595959',  # Dark Gray
    '#5F9ED1',  # Light Blue
    '#C85200',  # Dark Orange
    '#898989',  # Medium Gray
    '#A2C8EC',  # Pale Blue
    '#FFBC79',  # Light Orange
    '#CFCFCF',  # Light Gray
    '#009E73',  # Green (Colorblind-friendly)
    '#F0E442'   # Yellow (Colorblind-friendly)
]

CB_color_cycle_polar = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

markers = ['o', 'v', '^', 's', '+', '*', 'x', 'd', '<', '>', 'p']
linestyles = ['-', '--', '-.', ':', '-', '--', '-.', ':']

# MySQL connection parameters
mysql_config = {
    'user': 'handy',
    'password': 'handy',
    'host': 'localhost',
    'database': 'framework'
}

shots = 4000

mysql_config_online = {
    'user': 'handy',
    'password': 'handy',
    'host': 'ec2-16-171-135-24.eu-north-1.compute.amazonaws.com',
    'database': 'calibration_data'
}

import os
import sys

#module_path = os.path.abspath(os.path.join('..', 'functions'))
#if module_path not in sys.path:
#    sys.path.append(module_path)

from commons import (
    used_qubits, sum_middle_digits_dict
)

from commons import (Config, convert_utc_to_local, calculate_time_diff, get_count_1q, get_count_2q, 
    calculate_circuit_cost, get_correct_output_dict, calculate_success_rate_nassc, calculate_success_rate_tvd, 
    calculate_success_rate_polar, calculate_hellinger_distance, calculate_success_rate_tvd_new, 
    convert_to_json, is_mitigated, get_initial_mapping_json, normalize_counts, convert_dict_int_to_binary, reverse_string_keys, convert_dict_binary_to_int,
    sum_middle_digits_dict
)

from wrappers.multiprogramming_wrapper import (
    avoid_simultaneous_cnot, add_zz_on_simultaneous_cnot, 
    build_idle_coupling_map, multiprogram_compilation_qiskit, merge_circuits,
    get_LF_presets_cm
)
from wrappers.polar_wrapper import (
        polar_code_p2, get_logical_error_on_accepted_states, get_q1prep_sr, get_i_position, make_polar_qc_based_p2,
divide_half_list, get_q1prep_accepted_states
)

from wrappers.prune_wrapper import (
    create_full_graph, generate_figures, generate_node_errors, generate_edge_errors,
    get_latest_calibration_id, get_edges_threshold, get_readout_threshold, get_LF_qubits
)

from wrappers.dd_wrapper import (
    convert_dt_to_us, count_delay_durations, apply_pad_delay, get_delay_information, get_dd_information, 
    get_delay_and_dd_information_us
)

from wrappers.qiskit_wrapper import (
    apply_dd, get_zz_rates_from_backend_in_hz, get_qubits_T1_T2, get_gates_length, generate_errors_thermal_relaxation, 
generate_thermal_noise_model_on_used_qubits, get_neighbor_zz_rates_by_qubit, create_rzz_operator,
replace_delay_with_rzz, get_initial_layout_from_circuit
)

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

 
# # token = "476ea8c61cc54f36e4a21d70a8442f94203c9d87096eaad0886a3e8154d8c2e79bcad6f927c6050a76335dd68d783f478c1b828504748a4377b441c335c831aa"

# # # frisbee_among.0p@icloud.com
# # token = "04ede7f82299b792eae4daf3581f62415b3af676370f752f52d40e5851c7201d6e357c36e7bd55e4719c80d66bba7474412253f6c62f0ea7577bf056aa92eb62"

# # chores.acetone-0t@icloud.com
# token = "108bce3fd4c7a849d27b3eb41aad83f0c0a4f523cf6f3aa9735390405604613e51b9fc092722ef709554d9bb88a9dccd882eadeeadc590e554a8447fef8c8f97"
# QiskitRuntimeService.save_account(channel="ibm_quantum", token=token, overwrite=True)
# service = QiskitRuntimeService(channel="ibm_quantum", token=token)


token = "23OwipXqZzd8plKZR2LDMK-peWuU74UQAyjAhMIaFHCM"
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=token, instance="free", overwrite=True)
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token, instance="free")

hw_name = "ibm_brisbane"
# hw_name = "ibm_sherbrooke"
backend = service.backend(hw_name)

from qiskit.circuit import IfElseOp
backend.target.add_instruction(IfElseOp, name="if_else")

sim_ideal = AerSimulator()
sim_noisy = AerSimulator.from_backend(backend)

## change later back to 3
pm = generate_preset_pass_manager(
    optimization_level=3, backend=backend
    #, seed_transpiler=12345
)

def extract_z_syndrome(circuit, target, controls, measure):
    circuit.cx(controls[0], target)
    circuit.cx(controls[1], target)
    circuit.measure(target, measure)

    #with circuit.if_test((measure, True)):
    #    circuit.x(controls[0])

    #circuit.reset(target)

def extract_x_syndrome(circuit, target, controls, measure):
    circuit.h(target)
    circuit.cx(target, controls[0])
    circuit.cx(target, controls[1])
    circuit.h(target)
    circuit.measure(target, measure)





In [2]:
list_circuit_name = []
list_n = []
list_logical = []
polar_circuits = {}
polar_circuits_meas_data = {}
polar_circuits_x = {}
polar_circuits_x_meas_data = {}

print("-------- z -----------")
for i in range(3,7):
    list_circuit_name.append("polar_z_n{}".format(i))
    list_n.append(i)
    list_logical.append("0")
    polar_circuits[i] = (polar_code_p2(i, base="z"))
    polar_circuits_meas_data[i] = (polar_code_p2(i, meas_data=True, base="z"))
    #polar_circuits_x[i] = (polar_code_p2(i, base="x"))
    #polar_circuits_x_meas_data[i] = (polar_code_p2(i, meas_data=True, base="x"))

print("-------- x -----------")
for i in range(3,7):
    list_circuit_name.append("polar_x_n{}".format(i))
    list_n.append(i)
    list_logical.append("+")
    #polar_circuits[i] = (polar_code_p2(i, base="z"))
    #polar_circuits_meas_data[i] = (polar_code_p2(i, meas_data=True, base="z"))
    polar_circuits_x[i] = (polar_code_p2(i, base="x"))
    polar_circuits_x_meas_data[i] = (polar_code_p2(i, meas_data=True, base="x"))

-------- z -----------
n = 3 , b = 110 (3) , i = 4
n = 3 , b = 110 (3) , i = 4
n = 4 , b = 0011 (12) , i = 13
n = 4 , b = 0011 (12) , i = 13
n = 5 , b = 11100 (7) , i = 8
n = 5 , b = 11100 (7) , i = 8
n = 6 , b = 011010 (22) , i = 23
n = 6 , b = 011010 (22) , i = 23
-------- x -----------
n = 3 , b = 010 (2) , i = 3
n = 3 , b = 010 (2) , i = 3
n = 4 , b = 1101 (11) , i = 12
n = 4 , b = 1101 (11) , i = 12
n = 5 , b = 01100 (6) , i = 7
n = 5 , b = 01100 (6) , i = 7
n = 6 , b = 101010 (21) , i = 22
n = 6 , b = 101010 (21) , i = 22


In [3]:
def create_line_chart_combined(data, y, opt_values, metric, ax, type, ylabel, xlabel, yticks = None, 
                      y_bot = None, y_top = None, figsize = (12,8), c_idx = 0, reindex = None, 
                               x_index = "header_id", x_labels = [], i=0, hw_name = "", total_data = 0, log_val=False):
    tmp = ()
    
    for idx, opt in enumerate(opt_values):
        # print(idx, opt)
        idx_marker = idx + (len(opt_values) * i)
        # idx = idx 
        pivot = pd.pivot_table(data[data[y] == opt], 
                               values=[metric], 
                               index=x_index, 
                               columns=y, 
                               aggfunc='mean')    

        if len(pivot) == 0:
            continue

        # if hw_name == "brisbane" and opt == "na_triq_lcd" and i == 2:
        #     continue
        
        if reindex != 0:
            if reindex is None:
                if len(tmp) == 0:
                    tmp = pivot.index
                else:
                    pivot = pivot.reindex(tmp)
            else:
                pivot = pivot.reindex(reindex)

        idx_color = (idx % (len(markers)))
        # print(idx_color, idx, len(markers), idx_marker)

        # print (pivot)
         
        pivot.plot(kind='line', color=CB_color_cycle_polar[i], marker=markers[idx], linestyle=linestyles[idx], figsize=figsize, ax=ax)

        mean= pivot.mean().mean()

    if i == total_data-1:
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        
    
        if yticks is not None:
            ax.set_yticks(yticks)
    
        if y_bot is not None and y_top is not None:
            ax.set_ylim(bottom= y_bot, top= y_top)
    
        if x_labels is not None:
            ax.set_xticks(np.arange(len(x_labels)))
            ax.set_xticklabels(x_labels)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

        if log_val:
            ax.set_yscale("log")
    
        ax.grid()


def show_figure_by_opt_combined(dfs, y, y_values, metric, ylabel, xlabel, legend = None, legends = None, figsize=(10,6), reindex=None, 
                             x_index = "total_2q", x_labels = [], x_lim = [1, 100],
                            y_lim = [0,1], title=None, legend_position=None, circuit_name = "", hw_name = "",
                               log_val = False):
    # y_values = noise_levels

    metrics = [metric]
    row = 1
    col = 1

    fig, ax = plt.subplots(nrows=row, ncols=1)

    yticks = None
    
    for i in range(len(dfs)):
        # print(i)
        create_line_chart_combined(dfs[i], y, y_values, metrics[0], ax, y, ylabel, xlabel, yticks, 
                                   figsize=figsize, reindex=reindex,
                          x_index = x_index, x_labels=x_labels, i=i, hw_name=hw_name, total_data=len(dfs), log_val=log_val)

    # n=[]        
    # n.append(ax.axhline(np.NaN, color="gray", linestyle='-.'))

    # l1 = ax.legend(n, ["Mean"], loc=[0.82, 0.49])

    if legend is None:
            l2 = plt.legend()
    else:
        if legend_position == None:
            l2 = plt.legend(legends)
        else:
            l2 = plt.legend(legends, loc = legend_position)
    
    # plt.tight_layout()
    # ax.add_artist(l1)
    # plt.xticks(range(3,6))
    plt.xlim(x_lim) 
    plt.ylim(y_lim)
    plt.title(title)
    plt.savefig("./output/proposal_1/simulation/{}_{}_{}.png".format(circuit_name, hw_name, x_index), dpi=500, bbox_inches='tight')
    plt.show()
    
    return plt

def save_results_to_file(result_type, job_ids, service):
    for job_id in job_ids:
        job = service.job(job_id)
        result = job.result()
        
        with open(f"output/proposal_1/result_{result_type}_{job_id}.json", "w") as file:
            json.dump(result, file, cls=RuntimeEncoder)

def load_results_from_file(result_type):
    folder_path = f'output/proposal_1/'
    files = [f for f in os.listdir(folder_path) if f.startswith(f"result_{result_type}_")]
    results = []

    for file_name in files:
        with open(f"output/proposal_1/{file_name}", "r") as file:
            result = json.load(file, cls=RuntimeDecoder)

        results.append(result)

    return results

def save_result_to_file(result, hw_name, circuit_name, exp_type, error_scale, seed_simulator):
    with open(f"output/proposal_1/simulation/{hw_name}/{circuit_name}_{exp_type}_{error_scale}_{seed_simulator}.json", "w") as file:
        json.dump(result, file, cls=RuntimeEncoder)

def load_results_simulations_from_file(hw_name, circuit_name, exp_type, error_scale):
    folder_path = f'output/proposal_1/simulation/{hw_name}/'
    files = [f for f in os.listdir(folder_path) if f.startswith(f"{circuit_name}_{exp_type}_{error_scale}")]
    results = []

    for file_name in files:
        with open(f'output/proposal_1/simulation/{hw_name}/{file_name}', "r") as file:
            result = json.load(file, cls=RuntimeDecoder)

        results.append(result)

    return results



def generate_circuit_extraction_syndrome(k, meas_type, x_first = False):
    num_qbits = (2**k) + (2**(k-1))
    num_cbits = 2**(k-1)

    qc = QuantumCircuit(num_qbits, num_cbits)

    data_qubits = []
    ancillas = []
    for i in range(2**(k-1)):
        ancillas.append(i *3 + 2)

    for i in range(num_qbits):
        if i not in ancillas:
            data_qubits.append(i)

    d1, d2 = divide_half_list(data_qubits)
    #print(d1,d2, ancillas)

    for idx in range(len(d1)):

        if meas_type.lower() == "x":

            if x_first:
                qc.h(d1[idx])
                qc.cx(d1[idx], d2[idx])
            else:
                qc.h(ancillas[idx])
                qc.cx(ancillas[idx], d1[idx])
                qc.cx(ancillas[idx], d2[idx])
                qc.h(ancillas[idx])
                
                # qc.measure(ancillas[idx], idx)
                # qc.reset(ancillas[idx])

        elif meas_type.lower() == "z":
            qc.cx(d1[idx], ancillas[idx])
            qc.cx(d2[idx], ancillas[idx])

            # qc.measure(ancillas[idx], idx)
            # qc.reset(ancillas[idx])
    
    return qc

qc_x1 = generate_circuit_extraction_syndrome(1, "x", False)
qc_x1_first = generate_circuit_extraction_syndrome(1, "x", True)
qc_x2 = generate_circuit_extraction_syndrome(2, "x")
qc_x3 = generate_circuit_extraction_syndrome(3, "x")
qc_x4 = generate_circuit_extraction_syndrome(4, "x")
qc_x5 = generate_circuit_extraction_syndrome(5, "x")

qc_z2 = generate_circuit_extraction_syndrome(2, "z")
qc_z3 = generate_circuit_extraction_syndrome(3, "z")
qc_z4 = generate_circuit_extraction_syndrome(4, "z")
qc_z5 = generate_circuit_extraction_syndrome(5, "z")



#qc_x2.draw("mpl")




# Simulating Real-Time correction with Stim

In [5]:
polar_n3_x = polar_code_p2(3, False, "x", False)
# polar_n3_x.draw("mpl", fold=-1)

n = 3 , b = 010 (2) , i = 3


In [7]:
qasm_str = dumps(polar_n3_x)

#qasm_str = dumps(polar_n4_z)

In [8]:
import cirq
import stim
from cirq.contrib.qasm_import import circuit_from_qasm
import stimcirq

cirq_circuit = circuit_from_qasm(qasm_str)

stim_circuit = stimcirq.cirq_circuit_to_stim_circuit(cirq_circuit)

# n3_x
stim_circuit.append("M", [0,1,3,4,6,7,9,10])

# n4_z
#stim_circuit.append("M", [0,1,3,4,6,7,9,10,12,13,15,16,18,19,21,22])